# Multi-Agent Diverse and Private Synthetic QA Dataset Generation for RAG Evaluation

This notebook demonstrates the multi-agent framework for generating diverse and privacy-preserving synthetic QA datasets for RAG system evaluation, based on the research paper by Driouich et al. (2024).

## 📚 Framework Overview

The framework consists of three specialized agents:

1. **🔍 Diversity Agent**: Uses clustering techniques to maximize topical coverage and semantic variability
2. **🔒 Privacy Agent**: Detects and masks sensitive information across multiple domains
3. **🤖 QA Curation Agent**: Synthesizes private and diverse QA pairs suitable for RAG evaluation

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand how to use each agent independently
- Learn to combine agents for comprehensive QA dataset generation
- See how to evaluate diversity, privacy, and quality metrics
- Generate production-ready datasets for RAG evaluation


## 🛠️ Setup and Installation

First, let's install the required dependencies and set up our environment.


In [ ]:
# Install required dependencies
%pip install sentence-transformers scikit-learn spacy datasets matplotlib seaborn
!python -m spacy download en_core_web_sm


In [ ]:
# Import necessary libraries
import sys
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any

# Add SDG Hub to path
sys.path.append('../..')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

print("✅ Setup complete!")


## 📊 Sample Dataset Creation

Let's create a diverse sample dataset with various domains and privacy concerns to demonstrate the framework.

> **💡 For Large PDF Collections**: If you're processing large PDFs with Docling or similar tools, see the comprehensive integration guide at: `../../src/sdg_hub/flows/rag_evaluation/multi_agent_diverse_private/INPUT_SCHEMA.md`


In [ ]:
def create_sample_dataset() -> Dataset:
    """Create a diverse sample dataset for demonstration."""
    
    sample_data = {
        "document": [
            """
            Patient Sarah Johnson (DOB: 03/15/1985) was admitted to City General Hospital 
            on December 1st, 2024, with symptoms of chest pain and shortness of breath. 
            Her medical record number is MR-789456. Initial examination revealed elevated 
            blood pressure (160/95 mmHg) and irregular heartbeat. Dr. Michael Chen ordered 
            an ECG and blood tests. The patient's insurance ID is INS-456789123. 
            Treatment includes beta-blockers and lifestyle modifications.
            """,
            """
            TechCorp Inc. (NASDAQ: TECH) reported quarterly earnings of $3.2 million for Q4 2024, 
            representing an 18% increase from the previous quarter. CEO Jennifer Davis 
            (email: j.davis@techcorp.com, phone: 555-123-4567) announced plans for expansion 
            into the Asian market. The company's primary bank account (routing: 123456789, 
            account: 987654321) shows strong liquidity of $15.7 million.
            """,
            """
            In the landmark case Thompson vs. MegaCorp Industries (Case #CV-2024-005678), 
            the plaintiff alleges breach of contract regarding software delivery delays. 
            Attorney Lisa Rodriguez (Bar #12345, phone: 555-987-6543) represents the plaintiff, 
            while defendant is represented by Johnson & Associates. The contract, valued at 
            $2.5 million, specified delivery within 120 days.
            """,
            """
            The artificial intelligence research team at Stanford University published 
            groundbreaking findings on large language model performance in scientific domains. 
            The study, led by Dr. Amanda Chen and Dr. Robert Kim, analyzed 25 different models 
            across physics, chemistry, and biology tasks. Results showed that specialized 
            fine-tuning improved accuracy by 31% on average.
            """
        ],
        "domain": ["medical", "financial", "legal", "academic"],
        "document_id": ["med_001", "fin_002", "leg_003", "acad_004"]
    }
    
    return Dataset.from_dict(sample_data)

# Create the sample dataset
sample_dataset = create_sample_dataset()

print(f"📊 Created sample dataset with {len(sample_dataset)} documents")
print(f"🏷️ Domains: {set(sample_dataset['domain'])}")
print(f"📝 Columns: {sample_dataset.column_names}")

# Display first document preview
print("\n📖 Sample Document Preview:")
print(f"Domain: {sample_dataset['domain'][0]}")
print(f"Text: {sample_dataset['document'][0][:200]}...")


## 🔍 Part 1: Diversity Agent

The Diversity Agent uses semantic clustering to maximize topical coverage and ensure diverse question generation.


In [ ]:
# Import and initialize the Diversity Agent
from src.sdg_hub.core.blocks.rag_evaluation import DiversityAgentBlock

diversity_agent = DiversityAgentBlock(
    block_name="diversity_analysis",
    input_cols=["document", "domain"],
    output_cols=["diversity_cluster", "diversity_score", "semantic_features"],
    num_clusters=2,  # Small number for demo
    diversity_threshold=0.6,
    embedding_model="sentence-transformers/all-MiniLM-L6-v2"
)

print("🔍 Diversity Agent initialized!")
print(f"📊 Will create {diversity_agent.num_clusters} semantic clusters")

# Run diversity analysis
print("\n🚀 Running diversity analysis...")
diversity_result = diversity_agent(sample_dataset)

print("✅ Diversity analysis complete!")
print(f"📈 Average diversity score: {np.mean(diversity_result['diversity_score']):.3f}")

# Show results
for i in range(len(diversity_result)):
    print(f"\nDoc {i+1} ({diversity_result['domain'][i]}): Cluster {diversity_result['diversity_cluster'][i]}, Score: {diversity_result['diversity_score'][i]:.3f}")


## 🔒 Part 2: Privacy Agent

The Privacy Agent detects and masks sensitive information to ensure privacy compliance.


In [ ]:
# Import and initialize the Privacy Agent
from src.sdg_hub.core.blocks.rag_evaluation import PrivacyAgentBlock

privacy_agent = PrivacyAgentBlock(
    block_name="privacy_analysis",
    input_cols=["document", "domain"],
    output_cols=["privacy_masked_document", "privacy_entities", "privacy_score"],
    privacy_domains=["medical", "financial", "legal", "personal"],
    masking_strategy="entity_replacement",
    entity_types=["PERSON", "ORG", "GPE", "DATE", "MONEY", "PHONE", "EMAIL"]
)

print("🔒 Privacy Agent initialized!")
print(f"🛡️ Monitoring {len(privacy_agent.privacy_domains)} privacy domains")

# Run privacy analysis
print("\n🚀 Running privacy analysis...")
privacy_result = privacy_agent(sample_dataset)

print("✅ Privacy analysis complete!")
total_entities = sum(len(entities) for entities in privacy_result['privacy_entities'])
print(f"🔍 Detected {total_entities} sensitive entities")

# Show before/after examples
print("\n🎭 Privacy Masking Examples:")
for i in range(min(2, len(privacy_result))):
    print(f"\nDocument {i+1} ({privacy_result['domain'][i]}):")
    print(f"🔍 Entities: {len(privacy_result['privacy_entities'][i])}")
    print(f"📝 Original: {privacy_result['document'][i][:100]}...")
    print(f"🎭 Masked: {privacy_result['privacy_masked_document'][i][:100]}...")


## 🤖 Part 3: Combined Multi-Agent Analysis

Now let's combine both agents for comprehensive analysis and visualization.


In [ ]:
# Run combined analysis pipeline
print("🚀 Running combined multi-agent analysis...")

# Step 1: Diversity analysis
combined_result = diversity_agent(sample_dataset)

# Step 2: Privacy analysis on the diversity results
combined_result = privacy_agent(combined_result)

print("✅ Combined analysis complete!")
print(f"📊 Final dataset has {len(combined_result.column_names)} columns")

# Create analysis summary
analysis_df = pd.DataFrame({
    'domain': combined_result['domain'],
    'diversity_score': combined_result['diversity_score'],
    'privacy_score': combined_result['privacy_score'],
    'cluster': combined_result['diversity_cluster'],
    'num_entities': [len(entities) for entities in combined_result['privacy_entities']]
})

print("\n📊 Analysis Summary:")
print(analysis_df)


In [ ]:
# Create comprehensive visualization dashboard
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Diversity vs Privacy scatter
scatter = ax1.scatter(analysis_df['diversity_score'], analysis_df['privacy_score'], 
                     c=analysis_df['cluster'], cmap='viridis', s=100, alpha=0.7)
ax1.set_xlabel('Diversity Score')
ax1.set_ylabel('Privacy Score')
ax1.set_title('Diversity vs Privacy Analysis')
ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Privacy Threshold')
ax1.axvline(x=0.6, color='blue', linestyle='--', alpha=0.5, label='Diversity Threshold')
ax1.legend()

# Plot 2: Scores by domain
x = range(len(analysis_df))
width = 0.35
ax2.bar([i - width/2 for i in x], analysis_df['diversity_score'], width, label='Diversity', alpha=0.8)
ax2.bar([i + width/2 for i in x], analysis_df['privacy_score'], width, label='Privacy', alpha=0.8)
ax2.set_xlabel('Documents')
ax2.set_ylabel('Scores')
ax2.set_title('Diversity and Privacy Scores by Document')
ax2.set_xticks(x)
ax2.set_xticklabels(analysis_df['domain'], rotation=45)
ax2.legend()

# Plot 3: Entity detection by domain
ax3.bar(analysis_df['domain'], analysis_df['num_entities'], color='coral', alpha=0.7)
ax3.set_xlabel('Domain')
ax3.set_ylabel('Number of Entities')
ax3.set_title('Sensitive Entities Detected by Domain')
ax3.tick_params(axis='x', rotation=45)

# Plot 4: Quality assessment
high_quality = sum((analysis_df['diversity_score'] >= 0.6) & (analysis_df['privacy_score'] >= 0.5))
medium_quality = sum(((analysis_df['diversity_score'] >= 0.6) & (analysis_df['privacy_score'] < 0.5)) |
                    ((analysis_df['diversity_score'] < 0.6) & (analysis_df['privacy_score'] >= 0.5)))
low_quality = len(analysis_df) - high_quality - medium_quality

quality_counts = [high_quality, medium_quality, low_quality]
quality_labels = ['High Quality', 'Medium Quality', 'Low Quality']
colors = ['green', 'orange', 'red']
ax4.pie(quality_counts, labels=quality_labels, autopct='%1.1f%%', colors=colors)
ax4.set_title('Overall Quality Distribution')

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n📈 SUMMARY STATISTICS:")
print(f"Average Diversity Score: {analysis_df['diversity_score'].mean():.3f}")
print(f"Average Privacy Score: {analysis_df['privacy_score'].mean():.3f}")
print(f"Total Entities Detected: {analysis_df['num_entities'].sum()}")
print(f"High-Quality Documents: {high_quality}/{len(analysis_df)}")


## 🎯 Next Steps: QA Generation and Evaluation

This notebook demonstrated the core multi-agent analysis. To complete the RAG evaluation pipeline:

### 🚀 **QA Generation** (requires LLM configuration)
```python
# Configure your LLM API
from src.sdg_hub.core.blocks import LLMChatBlock, PromptBuilderBlock, TextParserBlock

# Use the analyzed data to generate diverse, private QA pairs
# See the full flow at: src/sdg_hub/flows/rag_evaluation/multi_agent_diverse_private/
```

### 📊 **Quality Evaluation**
- **Diversity Evaluation**: Assess semantic uniqueness of questions
- **Privacy Evaluation**: Verify compliance with privacy requirements  
- **Faithfulness Evaluation**: Check answer grounding in documents

### 💾 **Export Results**
```python
# Save analysis results
combined_result.to_json("multi_agent_analysis.json")
analysis_df.to_csv("analysis_summary.csv")
```

### 🎊 **Conclusion**
You've successfully used the Multi-Agent RAG Evaluation Framework to:
- ✅ Analyze semantic diversity across domains
- ✅ Detect and mask sensitive information
- ✅ Assess document quality for QA generation
- ✅ Visualize comprehensive analysis results

**Ready to generate diverse, privacy-compliant QA datasets for RAG evaluation!**
